# Modelos Ensemble Avanzados — Dry Bean Classification
**Proyecto Reto: Clasificación Multiclase — Grupo 5**

Este notebook compara los modelos base (Regresión Logística + Random Forest) con modelos ensemble avanzados:

1. Gradient Boosting (scikit-learn)
2. XGBoost
3. LightGBM

Se parte del dataset limpio generado en `01_eda.ipynb` y del preprocesamiento de `02_preprocesamiento_modelado.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_fscore_support
)
from sklearn.ensemble import (
    GradientBoostingClassifier, RandomForestClassifier
)
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost no disponible — se omitirá')

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('LightGBM no disponible — se omitirá')

pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
plt.style.use('dark_background')
sns.set_palette('Set2')

## 1. Cargar dataset y preparar splits

In [ ]:
df = pd.read_csv('../data/processed/dry_bean_clean.csv')
print('Shape:', df.shape)

X = df.drop('Class', axis=1)
y_raw = df['Class']

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_
print('Clases:', list(class_names))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}')

## 2. Definir y entrenar los modelos

In [ ]:
models = {
    'Regresión Logística': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, multi_class='multinomial'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=RANDOM_STATE
    ),
}

if HAS_XGB:
    models['XGBoost'] = XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=RANDOM_STATE, use_label_encoder=False,
        eval_metric='mlogloss', verbosity=0
    )

if HAS_LGBM:
    models['LightGBM'] = LGBMClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        random_state=RANDOM_STATE, verbose=-1
    )

print(f'Modelos a entrenar: {list(models.keys())}')

In [ ]:
results = {}

for name, model in models.items():
    print(f'\n--- {name} ---')
    
    # Usar X_train_sc para modelos que necesitan escalado, X_train para tree-based
    use_scaled = name in ['Regresión Logística']
    Xtr = X_train_sc if use_scaled else X_train
    Xte = X_test_sc if use_scaled else X_test
    
    model.fit(Xtr, y_train)
    
    train_pred = model.predict(Xtr)
    test_pred = model.predict(Xte)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    diff = abs(train_acc - test_acc) * 100
    
    # Validación cruzada
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(model, Xtr, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    
    results[name] = {
        'model': model,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'diff_pct': diff,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_pred': test_pred,
    }
    
    print(f'  Train: {train_acc:.4f}  |  Test: {test_acc:.4f}  |  Diff: {diff:.2f}%')
    print(f'  CV (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'  Overfitting: {"NO" if diff < 5 else "SÍ (>5%)"}')

## 3. Tabla comparativa

In [ ]:
comparison = pd.DataFrame({
    'Modelo': list(results.keys()),
    'Accuracy Train': [r['train_acc'] for r in results.values()],
    'Accuracy Test': [r['test_acc'] for r in results.values()],
    'Diferencia (%)': [r['diff_pct'] for r in results.values()],
    'CV Mean': [r['cv_mean'] for r in results.values()],
    'CV Std': [r['cv_std'] for r in results.values()],
})
comparison = comparison.sort_values('Accuracy Test', ascending=False).reset_index(drop=True)
comparison.style.highlight_max(
    subset=['Accuracy Train', 'Accuracy Test', 'CV Mean'], color='#22c55e'
).highlight_min(
    subset=['Diferencia (%)'], color='#22c55e'
).format(precision=4)

## 4. Gráfico comparativo de accuracies

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(comparison))
width = 0.35

bars1 = ax.bar(x_pos - width/2, comparison['Accuracy Train'], width, label='Train', color='#4ade80')
bars2 = ax.bar(x_pos + width/2, comparison['Accuracy Test'], width, label='Test', color='#60a5fa')

ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Comparación de Modelos — Train vs Test', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(comparison['Modelo'], rotation=15, ha='right')
ax.legend()
ax.set_ylim(0.85, 1.0)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../models/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en models/comparison_chart.png')

## 5. Mejor modelo — Matriz de confusión y reporte

In [ ]:
best_name = comparison.iloc[0]['Modelo']
best_result = results[best_name]
print(f'Mejor modelo: {best_name} (Test Accuracy: {best_result["test_acc"]:.4f})')

print('\nClassification Report:')
print(classification_report(y_test, best_result['y_pred'], target_names=class_names))

In [ ]:
cm = confusion_matrix(y_test, best_result['y_pred'])

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicho', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
ax.set_title(f'Matriz de Confusión — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance del mejor modelo

In [ ]:
best_model = best_result['model']

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': X.columns,
        'Importancia': importances
    }).sort_values('Importancia', ascending=True)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(feat_imp['Feature'], feat_imp['Importancia'], color='#4ade80')
    ax.set_xlabel('Importancia')
    ax.set_title(f'Feature Importance — {best_name}', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('../models/feature_importance_best.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'El modelo {best_name} no expone feature_importances_')

## 7. Guardar el mejor modelo

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')

print(f'Mejor modelo ({best_name}) guardado en ../models/best_model.pkl')
print('Scaler y LabelEncoder actualizados.')

## 8. Resumen

- Se entrenaron 5 modelos (o 3 si faltan XGBoost/LightGBM).
- Se compararon por accuracy, validación cruzada y brecha train/test.
- El mejor modelo se guardó como `best_model.pkl` junto con el scaler y label encoder.
- Los gráficos de comparación, confusión e importancia de features se guardaron en `models/`.